In [ ]:
"""Create list of already done jobs to exclude from rerun"""

import csv
import hashlib
import sys
from pathlib import Path
import json

# Load whitelist from JSON file
with open("RUNKEY_WHITELIST.json", "r") as f:
    WHITELIST = json.load(f)["whitelist"]

# CSV files passed as arguments
input_files = [
    "/Users/julian_dev/Projects/MasterThesis/QAOA/logs/ADAM/qaoa_results_ADAM_20260408_182901.csv",
    "/Users/julian_dev/Projects/MasterThesis/QAOA/logs/ADAM/qaoa_results_ADAM_20260408_151931.csv"
]
output_file = "completed_runs.txt"

def normalize(value):
    if value is None:
        return ""
    try:
        return json.dumps(value, sort_keys=True)
    except TypeError:
        return str(value).strip()

def build_run_key(row):
    """Create deterministic hash from whitelist params."""
    key_string = "|".join(
        f"{k}={normalize(row.get(k, ''))}" for k in WHITELIST
    )
    return hashlib.sha1(key_string.encode()).hexdigest()

seen = set()

for file in input_files:
    file = Path(file)
    if not file.exists():
        continue

    with open(file, newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            # Skip incomplete rows
            if "result" not in row or row["result"] in ("", "None"):
                continue

            run_key = build_run_key(row)
            print("PY:", "|".join(f"{k}={normalize(row.get(k, ''))}" for k in WHITELIST))
            seen.add(run_key)

# Write output
with open(output_file, "w") as f:
    for key in sorted(seen):
        f.write(key + "\n")

print(f"Collected {len(seen)} completed runs into {output_file}")

Collected 133 completed runs into completed_runs.txt


In [ ]:
def compare_qaoa_multi(files, group_cols=['n','m', 'p', 'precision/iterations'], require_common=True):
    import pandas as pd
    import os

    dfs = []
    for fp in files:
        label = os.path.splitext(os.path.basename(fp))[0]
        df = pd.read_csv(fp)
        df['source'] = label
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)

    # keep only configs present in all files (optional)
    if require_common and len(files) > 1:
        common = (
            df.groupby(group_cols)['source']
            .nunique()
            .reset_index()
        )
        common = common[common['source'] == len(files)][group_cols]
        df = df.merge(common, on=group_cols, how='inner')

    stats = (
        df.groupby('source')['approx_ratio']
        .agg(
            mean='mean',
            median='median',
            p25=lambda x: x.quantile(0.25),
            p75=lambda x: x.quantile(0.75),
            std='std',
            count='count'
        )
        .reset_index()
    )

    return stats

In [11]:
files = [
    "logs/ADAM/qaoa_results_ADAM_20260415_081141_complete.csv",
    "logs/ADAM/qaoa_results_ADAM_20260413_213043_complete_closeToZero.csv"
]

stats = compare_qaoa_multi(files)
print(stats)

                                              source      mean    median  \
0  qaoa_results_ADAM_20260413_213043_complete_clo...  0.767832  0.780811   
1         qaoa_results_ADAM_20260415_081141_complete  0.767045  0.779544   

        p25       p75       std  count  
0  0.731756  0.823028  0.064993    192  
1  0.732553  0.817814  0.064300    192  


In [ ]:
files = [
    "logs/COBYLA/qaoa_results_COBYLA_20260414_190238.csv",
    "logs/COBYLA/qaoa_results_COBYLA_20260415_080702.csv"
]

stats = compare_qaoa_multi(files)
print(stats)

                                              source      mean    median  \
0  qaoa_results_ADAM_20260410_144026_line_sameIni...  0.680276  0.683058   

        p25       p75       std  count  
0  0.666667  0.694095  0.029639    203  


In [ ]:
files = [
    "logs/ADAM/qaoa_results_ADAM_20260410_144026_line_sameInitialParams.csv",
]

stats = compare_qaoa_multi(files)
print(stats)